# Phase-3 train

One run, then DEV sanity. Fresh T4 runtime. Copy the frozen `phase3/IMPLEMENTATION_MANIFEST.md` to `MyDrive/Hinglish_Phase3/IMPLEMENTATION_MANIFEST.md` before launch. All attempts and recoveries persist. Set ARM/SEED to the plan; never resume partial training. DEV cannot authorize a retry.


In [ ]:
import base64, hashlib, io, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path
from google.colab import drive, files
import importlib.metadata
drive.mount('/content/drive')
STORE = Path('/content/drive/MyDrive/Hinglish_Phase3')
STORE.mkdir(parents=True, exist_ok=True)
PACKAGE = Path('/content/phase3_step2')
PACKAGE.mkdir(parents=True, exist_ok=True)
TRAIN_ROOT = STORE / 'train'
# Paste the SHA-256 printed by build_implementation_manifest.py from the archived release.
# It cannot be embedded in a notebook which that same manifest hashes.
EXPECTED_IMPLEMENTATION_SHA256 = ''
companion = STORE / 'IMPLEMENTATION_MANIFEST.md'
remedy = ('Copy the matching phase3/IMPLEMENTATION_MANIFEST.md to '
          'MyDrive/Hinglish_Phase3/IMPLEMENTATION_MANIFEST.md and paste its archived '
          'build SHA-256 into EXPECTED_IMPLEMENTATION_SHA256, then rerun this cell.')
assert companion.is_file(), 'Missing implementation manifest. ' + remedy
assert len(EXPECTED_IMPLEMENTATION_SHA256) == 64, 'Expected manifest SHA-256 is unset. ' + remedy
assert hashlib.sha256(companion.read_bytes()).hexdigest() == EXPECTED_IMPLEMENTATION_SHA256, 'Wrong implementation manifest. ' + remedy
assert importlib.metadata.version('torch') == '2.11.0+cu130', 'Wrong Colab image: stop; never install torch.'
BLOB = base64.b64decode('...')
assert hashlib.sha256(BLOB).hexdigest() == '77f17e89f4c77b3940246c254be72617b12d3864875b531ccfb50152efa94883'
with zipfile.ZipFile(io.BytesIO(BLOB)) as z:
    assert not any('final_test' in n.lower() for n in z.namelist())
with zipfile.ZipFile(io.BytesIO(BLOB)) as archive:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in archive.namelist())
    archive.extractall(PACKAGE)
sys.path.insert(0, str(PACKAGE))
os.chdir(PACKAGE)
contract = json.loads((PACKAGE / 'phase3/step2_runtime_contract.json').read_bytes())
for path, checksum in contract['files'].items():
    assert hashlib.sha256((PACKAGE / path).read_bytes()).hexdigest() == checksum, path
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet'] + [f'{k}=={v}' for k,v in contract['dependencies'].items() if k != 'torch'], check=True)
from t4_smoke import train_smoke as ts, train_phase3 as train, phase3_eval as ev, phase3_contract as c
assert not ts.stack_mismatches(ts.versions()), 'Stack mismatch; restart runtime if imports are stale.'
import torch
assert torch.cuda.is_available() and torch.cuda.device_count() == 1
assert 'T4' in torch.cuda.get_device_name(0) and torch.cuda.get_device_capability(0) == (7, 5)
# Copy the authenticated companion even when a stale local copy exists.
if (STORE / 'IMPLEMENTATION_MANIFEST.md').is_file():
    shutil.copyfile(STORE / 'IMPLEMENTATION_MANIFEST.md', PACKAGE / 'phase3/IMPLEMENTATION_MANIFEST.md')
c.verify_implementation(PACKAGE)
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
except Exception:
    pass


In [ ]:
# Operator settings must match plan: slot 11 then 22 then 33, EC -> ER -> EE.
RECOVER_STALE_WRITER = False  # True only after confirming no other trainer session is active.
RECOVER_STALE_LOCK = False    # True only after confirming no other evaluator is active.
print('Training writer lock:', train.p2.read_writer_lock(TRAIN_ROOT))
if RECOVER_STALE_WRITER:
    train.recover_stale_writer(TRAIN_ROOT, confirmed_no_other_session=True)
pending = train.p2.next_run(train.p2.plan(train.p2.read_log(TRAIN_ROOT / train.p2.ATTEMPT_LOG)))
print('Pending plan (arm/seed/attempt):', pending)
from t4_smoke import phase3_registry as registry
try:
    with registry.locked(TRAIN_ROOT):
        print('Evaluation registry lock: available')
except train.p2.Refused:
    print('Evaluation registry lock: held or unsupported')
    raise
print('Evaluation engine locks:', [str(p) for p in STORE.rglob('RUNNING.lock')])
ARM = pending['arm'] if pending['kind'] == 'train' else None  # Operator may override; trainer asserts the plan.
SEED = pending['seed'] if pending['kind'] == 'train' else None
common = ['--root', str(PACKAGE), '--train-root', str(TRAIN_ROOT)]
def evaluate(folder, args):
    return subprocess.run([sys.executable, '-m', 't4_smoke.phase3_eval'] + args).returncode
try:
    ev.run_dev_sanity(TRAIN_ROOT, PACKAGE, recover=RECOVER_STALE_LOCK, evaluate=evaluate)
    if pending['kind'] == 'complete':
        print('Training plan and DEV sanity complete; proceed to the adapter freeze.')
    else:
        result = subprocess.run([sys.executable, '-m', 't4_smoke.train_phase3', 'train'] + common + ['--arm', ARM, '--seed', str(SEED)])
        if result.returncode == 0:
            ev.run_dev_sanity(TRAIN_ROOT, PACKAGE, recover=RECOVER_STALE_LOCK, evaluate=evaluate)
        else:
            print('Training exit code:', result.returncode, '; inspect attempt plan/status. No automatic retry.')
finally:
    archive = shutil.make_archive(str(STORE / f'Phase3_Train_{ARM}-s{SEED}'), 'zip', TRAIN_ROOT)
    files.download(archive)
